In [ ]:

# Task 3: Netflix Audience Rating Classification


import os
import warnings

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

RANDOM_STATE = 42
GROUP_ORDER = ["Kids", "Family", "Teen", "Adult"]



# 1. Load and inspect the dataset


df = pd.read_csv("data/Dataset.csv")

required_columns = [
    "title", "type", "release_year", "rating",
    "duration", "date_added", "country", "listed_in"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df.drop_duplicates(
    subset=["title", "type", "release_year"]
).reset_index(drop=True)

print("Total rows:", len(df))
print("\nOriginal rating distribution:")
print(df["rating"].value_counts(dropna=False))

df["rating"].value_counts().plot(kind="bar", figsize=(9, 4))
plt.title("Netflix Rating Distribution")
plt.xlabel("Original rating")
plt.ylabel("Number of titles")
plt.tight_layout()
plt.savefig("figures/task3_rating_distribution.png", dpi=150)
plt.close()






rating_groups = {
    "TV-Y": "Kids",
    "TV-Y7": "Kids",
    "TV-Y7-FV": "Kids",
    "TV-G": "Kids",
    "G": "Kids",

    "PG": "Family",
    "TV-PG": "Family",

    "PG-13": "Teen",
    "TV-14": "Teen",

    "R": "Adult",
    "TV-MA": "Adult",
    "NC-17": "Adult"
}

df["group"] = df["rating"].map(rating_groups)

# Exclude unrated and unsupported rating categories.
model_df = df.dropna(subset=["group"]).copy()
model_df["group"] = pd.Categorical(
    model_df["group"], categories=GROUP_ORDER
)

print("\nGrouped rating distribution:")
print(model_df["group"].value_counts().reindex(GROUP_ORDER))



# 3. Feature preparation functions

def split_genres(value):
    if pd.isna(value):
        return []

    return [
        genre.strip()
        for genre in str(value).split(",")
        if genre.strip()
    ]


def get_main_country(data):
    country = (
        data["country"]
        .fillna("Unknown")
        .astype(str)
        .str.split(",")
        .str[0]
        .str.strip()
    )

    country = country.replace(
        ["", "Not Given", "nan"], "Unknown"
    )

    return country


def make_numeric_features(data):
    features = pd.DataFrame(index=data.index)

    features["release_year"] = pd.to_numeric(
        data["release_year"], errors="coerce"
    )

    features["year_added"] = pd.to_datetime(
        data["date_added"], errors="coerce"
    ).dt.year

    features["is_tv_show"] = (
        data["type"].fillna("") == "TV Show"
    ).astype(int)

    # Duration is deliberately excluded to reduce shortcut learning.
    return features



# 4. Custom transformer: learn preprocessing from
#    the training fold only
class NetflixFeatureBuilder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = X.copy()

        # Learn genre vocabulary from this training fold only.
        self.genre_encoder_ = MultiLabelBinarizer()
        self.genre_encoder_.fit(
            X["listed_in"].apply(split_genres)
        )

        # Learn frequent countries from this training fold only.
        country = get_main_country(X)
        self.top_countries_ = (
            country.value_counts().head(15).index.tolist()
        )

        self.country_categories_ = (
            self.top_countries_ + ["Other"]
        )

        # Learn numeric imputation values from this fold only.
        numeric = make_numeric_features(X)
        self.numeric_medians_ = numeric.median().fillna(0)

        self.feature_names_out_ = (
            ["genre__" + str(g)
             for g in self.genre_encoder_.classes_]
            + ["country__" + str(c)
               for c in self.country_categories_]
            + list(numeric.columns)
        )

        return self

    def transform(self, X):
        X = X.copy()

        # Genre multi-hot encoding.
        genre_values = self.genre_encoder_.transform(
            X["listed_in"].apply(split_genres)
        )

        genre_df = pd.DataFrame(
            genre_values,
            columns=[
                "genre__" + str(g)
                for g in self.genre_encoder_.classes_
            ],
            index=X.index
        )

        # Country encoding with a fixed training-fold vocabulary.
        country = get_main_country(X)
        country = country.where(
            country.isin(self.top_countries_), "Other"
        )

        country_df = pd.DataFrame(
            {
                "country__" + str(c):
                    (country == c).astype(int)
                for c in self.country_categories_
            },
            index=X.index
        )

        # Numeric features and training-fold medians.
        numeric_df = make_numeric_features(X)
        numeric_df = numeric_df.fillna(self.numeric_medians_)

        features = pd.concat(
            [genre_df, country_df, numeric_df], axis=1
        )

        return features.reindex(
            columns=self.feature_names_out_, fill_value=0
        )

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_



# 5. Train/test split

train_df, test_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=model_df["group"]
)

X_train = train_df.drop(columns=["group", "rating"])
X_test = test_df.drop(columns=["group", "rating"])

y_train = train_df["group"].astype(str)
y_test = test_df["group"].astype(str)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))



# 6. Build pipelines and compare models

models = {
    "Baseline": Pipeline([
        ("features", NetflixFeatureBuilder()),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),

    "Logistic Regression": Pipeline([
        ("features", NetflixFeatureBuilder()),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Decision Tree": Pipeline([
        ("features", NetflixFeatureBuilder()),
        ("model", DecisionTreeClassifier(
            max_depth=8,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Random Forest": Pipeline([
        ("features", NetflixFeatureBuilder()),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(
            y_test, predictions,
            labels=GROUP_ORDER,
            average="macro",
            zero_division=0
        )
    })



# 7. Tune Random Forest using training data only


parameter_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [10, 20, None],
    "model__min_samples_leaf": [1, 3]
}

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

search = GridSearchCV(
    estimator=Pipeline([
        ("features", NetflixFeatureBuilder()),
        ("model", RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1
        ))
    ]),
    param_grid=parameter_grid,
    cv=cv_strategy,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
best_predictions = best_model.predict(X_test)

results.append({
    "Model": "Tuned Random Forest",
    "Accuracy": accuracy_score(y_test, best_predictions),
    "Macro F1": f1_score(
        y_test, best_predictions,
        labels=GROUP_ORDER,
        average="macro",
        zero_division=0
    )
})

results_df = (
    pd.DataFrame(results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
    .round(3)
)

print("\nBest Random Forest settings:")
print(search.best_params_)

print("\nBest cross-validation macro F1:")
print(round(search.best_score_, 3))

print("\nModel comparison:")
print(results_df.to_string(index=False))

results_df.to_csv(
    "results/task3_model_comparison.csv", index=False
)



# 8. Evaluate the tuned model on the held-out test set


report = classification_report(
    y_test,
    best_predictions,
    labels=GROUP_ORDER,
    zero_division=0
)

print("\nClassification report:")
print(report)

with open(
    "results/task3_classification_report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write("Best parameters:\n")
    file.write(str(search.best_params_) + "\n\n")
    file.write(
        f"Cross-validation macro F1: "
        f"{search.best_score_:.4f}\n\n"
    )
    file.write(report)

# Confusion matrix.
fig, ax = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    labels=GROUP_ORDER,
    display_labels=GROUP_ORDER,
    cmap="Blues",
    colorbar=False,
    ax=ax
)

ax.set_title("Netflix Audience Rating Classification")
plt.tight_layout()
plt.savefig(
    "figures/task3_confusion_matrix.png", dpi=150
)
plt.close()



# 9. Feature importance


fitted_features = best_model.named_steps["features"]
fitted_forest = best_model.named_steps["model"]

importance = pd.Series(
    fitted_forest.feature_importances_,
    index=fitted_features.get_feature_names_out()
).sort_values(ascending=False)

top_importance = importance.head(12)

print("\nTop 12 features:")
print(top_importance.to_string())

top_importance.sort_values().plot(
    kind="barh", figsize=(9, 5)
)

plt.title("Top 12 Features Used by Random Forest")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.savefig(
    "figures/task3_feature_importance.png", dpi=150
)
plt.close()

importance.rename("importance").to_csv(
    "results/task3_feature_importance.csv"
)



# 10. Compare grouped ratings with original ratings


rating_counts = df["rating"].value_counts()

eligible_ratings = [
    rating
    for rating, count in rating_counts.items()
    if count >= 10 and rating in rating_groups
]

comparison_df = df[
    df["rating"].isin(eligible_ratings)
].copy()

comparison_df["group"] = comparison_df["rating"].map(
    rating_groups
)

raw_train, raw_test = train_test_split(
    comparison_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=comparison_df["rating"]
)

raw_X_train = raw_train.drop(
    columns=["rating", "group"]
)
raw_X_test = raw_test.drop(
    columns=["rating", "group"]
)

# FIX: Extract parameters by removing the pipeline prefix.
best_rf_params = {
    key.replace("model__", ""): value
    for key, value in search.best_params_.items()
    if key.startswith("model__")
}

# Add shared Random Forest settings.
best_rf_params.update({
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1
})

# Model 1: predict the four grouped rating categories.
group_model = Pipeline([
    ("features", NetflixFeatureBuilder()),
    ("model", RandomForestClassifier(**best_rf_params))
])

group_model.fit(raw_X_train, raw_train["group"])
group_predictions = group_model.predict(raw_X_test)

# Model 2: predict the original rating categories.
original_model = Pipeline([
    ("features", NetflixFeatureBuilder()),
    ("model", RandomForestClassifier(**best_rf_params))
])

original_model.fit(raw_X_train, raw_train["rating"])
original_predictions = original_model.predict(raw_X_test)

comparison_results = pd.DataFrame([
    {
        "Version": "Original ratings",
        "Classes": raw_train["rating"].nunique(),
        "Accuracy": accuracy_score(
            raw_test["rating"], original_predictions
        ),
        "Macro F1": f1_score(
            raw_test["rating"],
            original_predictions,
            labels=sorted(eligible_ratings),
            average="macro",
            zero_division=0
        )
    },
    {
        "Version": "Grouped ratings",
        "Classes": 4,
        "Accuracy": accuracy_score(
            raw_test["group"], group_predictions
        ),
        "Macro F1": f1_score(
            raw_test["group"],
            group_predictions,
            labels=GROUP_ORDER,
            average="macro",
            zero_division=0
        )
    }
]).round(3)

print("\nOriginal vs grouped rating comparison:")
print(comparison_results.to_string(index=False))

comparison_results.to_csv(
    "results/task3_grouping_comparison.csv",
    index=False
)

print("\nTask 3 completed successfully.")
print("Charts saved in: figures/")
print("Reports and CSV files saved in: results/")


Total rows: 8787

Original rating distribution:
rating
TV-MA       3205
TV-14       2155
TV-PG        861
R            798
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

Grouped rating distribution:
group
Kids       906
Family    1148
Teen      2645
Adult     4006
Name: count, dtype: int64

Training rows: 6964
Testing rows: 1741

Best Random Forest settings:
{'model__max_depth': None, 'model__min_samples_leaf': 3, 'model__n_estimators': 400}

Best cross-validation macro F1:
0.608

Model comparison:
              Model  Accuracy  Macro F1
Tuned Random Forest     0.605     0.585
Logistic Regression     0.591     0.583
      Random Forest     0.599     0.566
      Decision Tree     0.508     0.508
           Baseline     0.460     0.158

Classification report:
              precision    recall  f1-score   support

        Kids       0.72    